# OpenPlaque — Multi-Seed RCA-Control Adjudication v2.1
Post-hoc control-selection fix. Blind v2.0 acceptance is frozen; only the RCA positive-control adjudication order is corrected.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/OpenPlaque'
OUTPUT_DIR = DRIVE_ROOT + '/Left_Coronary_Source_Ostium_Multiseed_Control_Adjudication_v2_1'
BRANCH = 'left-coronary-multiseed-control-fix-from-main'
PINNED_SCIENCE_COMMIT = '39514a36ce3df8b55629c448a4e22251abcbb69e'
BASELINE = '0593b453959f5a353d644267fbeef24b514ef4d7'
EXPECTED_ALGORITHM = 'left-coronary-source-ostium-multiseed-v2.1-control-adjudication'
print('Branch:', BRANCH)
print('Pinned science commit:', PINNED_SCIENCE_COMMIT)
print('Output:', OUTPUT_DIR)


In [ ]:
import os, shutil
repo='/content/OpenPlaque'
if os.path.exists(repo): shutil.rmtree(repo)
!git clone --depth 20 --branch $BRANCH https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!git -C /content/OpenPlaque checkout --detach $PINNED_SCIENCE_COMMIT
HEAD = !git -C /content/OpenPlaque rev-parse HEAD
HEAD = HEAD[0].strip()
print('Checked out HEAD:', HEAD)
assert HEAD == PINNED_SCIENCE_COMMIT, (HEAD, PINNED_SCIENCE_COMMIT)
merge_base = !git -C /content/OpenPlaque merge-base HEAD $BASELINE
print('Merge base:', merge_base[0] if merge_base else 'missing')
assert merge_base and merge_base[0].strip() == BASELINE


In [ ]:
%pip uninstall -y openplaque >/dev/null 2>&1
%pip install -q --no-cache-dir --force-reinstall --no-deps /content/OpenPlaque
%pip install -q pytest SimpleITK scipy pandas matplotlib numpy pydicom psutil


In [ ]:
import sys, importlib, pathlib, pytest
for name in list(sys.modules):
    if name == 'openplaque' or name.startswith('openplaque.'):
        del sys.modules[name]
importlib.invalidate_caches()
import openplaque
from openplaque import left_coronary_source_ostium_multiseed_control_v2_1 as exp
print('openplaque:', openplaque.__file__)
print('experiment:', exp.__file__)
print('algorithm:', exp.ALGORITHM)
assert exp.BASELINE == BASELINE
assert exp.ALGORITHM == EXPECTED_ALGORITHM
compile(pathlib.Path(exp.__file__).read_text(), exp.__file__, 'exec')
test_file='/content/OpenPlaque/tests/test_left_coronary_source_ostium_multiseed_control_v2_1.py'
rc = pytest.main(['-q', test_file])
if rc != 0: raise RuntimeError(f'pytest failed with exit code {rc}')


In [ ]:
from pathlib import Path
root=Path(DRIVE_ROOT)
required=[
 root/'Left_Coronary_Source_Ostium_Multiseed_v2/summary.json',
 root/'Left_Coronary_Source_Ostium_Multiseed_v2/blind_multiseed_root_hypotheses.csv',
 root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy',
 root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',
 root/'Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',
 root/'PCAT_RCA_10_50/rca_centerline_smoothed_zyx.csv',
 root/'Cache/LAD_Frozen_Proximal_Reacquisition_v1/combined_lad_centerline.csv',
 root/'Joint_Three_Vessel_Template_Classifier_v1/candidate_04_source_path.csv',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/aorta.nii.gz',
 root/'Cache/Left_Coronary_Source_Ostium_Discovery_v1_2/root_vesselness.npy',
 root/'Cache/Left_Coronary_Source_Ostium_Discovery_v1_2/root_vesselness_meta.json',
]
missing=[str(p) for p in required if not p.exists()]
print('Preflight required:',len(required),'missing:',len(missing))
if missing: raise FileNotFoundError('\n'.join(missing))


In [ ]:
import gc, time
from openplaque.left_coronary_source_ostium_multiseed_control_v2_1 import run
gc.collect()
t0=time.time()
result = run(DRIVE_ROOT, OUTPUT_DIR)
s=result['summary']
print('ELAPSED MIN:', round((time.time()-t0)/60,2))
print('STATUS:', s['status'])
print('INPUT BLIND-ACCEPTED:', s.get('input_blind_accepted_hypotheses'))
print('RCA CONTROL PASS:', s.get('RCA_control_pass'))
print('RCA CONTROL:', s.get('RCA_control'))
print('SECOND CORONARY:', s.get('second_coronary_candidate'))
print('REPORT:', result.get('report'))
print('ZIP:', result.get('zip'))
